In [11]:
import pandas as pd

# 생략 없이 모두 출력하도록 판다스 옵션 변경
pd.set_option('display.max_rows', None)      # 모든 행 출력
pd.set_option('display.max_columns', None)   # 모든 열 출력
pd.set_option('display.max_colwidth', None)  # 긴 텍스트 내용 잘림 방지

# 전처리된 파일 불러오기
df = pd.read_csv('payment_data_ko_preprocessed.csv')

# 2. 파이썬이 인식하는 실제 컬럼 목록 출력
print("현재 파일에 존재하는 컬럼 목록:", df.columns.tolist())

df.to_excel('payment_data_ko_preprocessed.xlsx', index=False)

# 데이터 출력
print(df)

현재 파일에 존재하는 컬럼 목록: ['플래그', '문의 내용', '카테고리', '의도', '응답']
           플래그                                          문의 내용 카테고리        의도  \
0           BZ                   청구서를 보여주세요{{Invoice Number}}   결제    청구서 확인   
1        BLMQZ                     {name}에서 보낸 청구서를 확인해야 합니다.   결제    청구서 확인   
2          BCL              {name}에서 보낸 청구서를 확인해야 합니다. 도와주세요.   결제    청구서 확인   
3          BIL            {name}에서 보낸 청구서를 보는 데 도움을 주실 수 있나요?   결제    청구서 확인   
4          BIL                       송장 #37777을 어떻게 찾을 수 있나요?   결제    청구서 확인   
5         BLMW                          {name}에서 보낸 송장을 찾아야 해   결제    청구서 확인   
6         BLQZ                   을 잠깐 살펴보는 데 도움이 필요합니다 #37777   결제    청구서 확인   
7          BIL                청구서 #12588을 확인하는 것을 도와주실 수 있나요?   결제    청구서 확인   
8           BL                              청구서 #12588을 찾아보세요   결제    청구서 확인   
9          BLQ               청구서를 간략히 살펴보는 데 도움이 필요합니다 #85632   결제    청구서 확인   
10          BL                     청구서 #85632를 찾는 데 도움이 필요합니다   

In [ ]:
import pandas as pd
import re
import json

# ==========================================
# 0. 데이터 불러오기
# ==========================================
print("데이터 전처리를 시작합니다...")
df = pd.read_csv('cs2_data_ko.csv')

initial_count = len(df)
df = df.dropna(subset=['문의 내용', '응답'])
final_count = len(df)

if initial_count != final_count:
    print(f"결측치가 발견되어 삭제하였습니다.{initial_count - final_count}건의 데이터를 삭제했습니다.")
else:
    print("결측치가 발견되지 않았습니다.")

# ==========================================
# 1 & 2단계: 텍스트 정제 함수 정의 및 적용
# ==========================================
def clean_placeholders(text):
    text = str(text)
    # 주요 템플릿 변수 치환
    text = re.sub(r'\{\{Order Number\}\}', 'ORD-12345', text)
    text = re.sub(r'\{\{Customer Support Hours\}\}', '평일 09:00~18:00', text)
    text = re.sub(r'\{\{Online Company Portal Info\}\}', '공식 홈페이지', text)
    # 나머지 템플릿 변수 일괄 치환
    text = re.sub(r'\{\{.*?\}\}', 'OOO', text)
    return text

def clean_text(text):
    # 연속된 공백 하나로 축소 및 양끝 공백 제거
    text = re.sub(r'\s+', ' ', text).strip()
    return text


print("템플릿 기호 및 공백을 정제하고 있습니다...")
df['문의 내용'] = df['문의 내용'].apply(clean_placeholders).apply(clean_text)
df['응답'] = df['응답'].apply(clean_placeholders).apply(clean_text)

# ==========================================
# 3단계: 목적에 맞는 데이터셋 추출 (파인튜닝용)
# ==========================================
# 전체 내용 전처리할때
# qa_df = df[['문의 내용', '응답']]

# 카테고리 열의 값이 결제인 데이터(행)만 필터링한다.
payment_df = df[df['카테고리'] == '결제']
# 필터링된 데이터에서 파인튜닝에 필요한 두가지 열만 최종 추출
qa_df = payment_df[['문의 내용', '응답']]


# ==========================================
# 4단계: JSONL 파일로 저장
# ==========================================
output_filename = 'processed_cs_data.jsonl'
print(f"데이터를 {output_filename} 파일로 저장합니다...")

with open(output_filename, 'w', encoding='utf-8') as f:
    for _, row in qa_df.iterrows():
        json_obj = {
            "instruction": row['문의 내용'],
            "output": row['응답']
        }
        f.write(json.dumps(json_obj, ensure_ascii=False) + '\n')

print("데이터 전처리 및 저장이 완료되었습니다.\n")

# ==========================================
# 5. 결과 확인하기 (상위 3건 출력)
# ==========================================
print("--- 전처리 완료 데이터 확인 (상위 10건) ---")
with open(output_filename, 'r', encoding='utf-8') as f:
    for i in range(10):
        line = f.readline()
        if not line:
            break
        data = json.loads(line)
        print(f"질문: {data['instruction']}")
        print(f"답변: {data['output']}\n")

데이터 전처리를 시작합니다...
결측치가 발견되지 않았습니다.
템플릿 기호 및 공백을 정제하고 있습니다...
데이터를 processed_cs_data.jsonl 파일로 저장합니다...
데이터 전처리 및 저장이 완료되었습니다.

--- 전처리 완료 데이터 확인 (상위 3건) ---
질문: 청구서를 보여주세요OOO
답변: 송장 번호 OOO로 청구서를 찾으셔야 한다는 점을 이해합니다. 제가 도와드리겠습니다. 잠시 시간을 내어 기록을 확인하고 필요한 정보를 찾아보시기 바랍니다.

질문: OOO에서 보낸 청구서를 확인해야 합니다.
답변: 물론이죠! OOO에서 보낸 인보이스를 보려고 하신 것으로 알고 있습니다. 추가 지원을 위해 해당 인보이스에 관련된 특정 날짜 범위나 기타 관련 세부정보를 알려주시겠습니까? 이렇게 하면 송장을 정확하고 신속하게 찾는 데 도움이 됩니다. 안심해 주세요. 저는 고객님께 절차를 안내하고 고객님이 필요한 정보에 접근할 수 있도록 도와드리기 위해 왔습니다.

질문: OOO에서 보낸 청구서를 확인해야 합니다. 도와주세요.
답변: 저는 그것을하고있다! OOO OOO, OOO의 송장을 긴급하게 확인해야 한다는 점을 이해합니다. 즉시 도움을 드릴 수 있도록 OOO와 같은 추가 정보나 송장과 관련된 특정 세부정보를 제공해 주시겠습니까? 이를 통해 보다 정확하고 효율적인 검색을 보장할 수 있습니다. 안심해 주세요. 저는 고객님께 절차를 안내하고 그 과정에서 고객님이 가질 수 있는 모든 우려 사항을 해결하기 위해 왔습니다.

질문: OOO에서 보낸 청구서를 보는 데 도움을 주실 수 있나요?
답변: 물론이죠! OOO OOO의 송장을 보는 데 도움이 필요하다는 점을 이해합니다. 도움을 요청해주셔서 감사합니다. 계속 진행하여 정확한 송장 세부정보를 제공하려면 OOO 또는 고객님이 갖고 있는 기타 관련 정보를 제공해 주시겠습니까? 저는 고객님께 모든 단계를 안내하고 원활한 경험을 보장하기 위해 왔습니다.

질문: 송장 #37777을 어떻게 찾을 수 있나요?
답변: 

In [13]:
import pandas as pd

df = pd.read_csv('cs2_data_ko.csv')
payment_df = df[df['카테고리'] == '결제']

payment_df

,플래그,문의 내용,카테고리,의도,응답
3918,BZ,청구서를 보여주세요{{Invoice Number}},결제,청구서 확인,송장 번호 {{Invoice Number}}로 청구서를 찾으셔야 한다는 점을 이해합...
3919,BLMQZ,{{Person Name}}에서 보낸 청구서를 확인해야 합니다.,결제,청구서 확인,물론이죠! {{Person Name}}에서 보낸 인보이스를 보려고 하신 것으로 알고...
3920,BCL,{{Person Name}}에서 보낸 청구서를 확인해야 합니다. 도와주세요.,결제,청구서 확인,저는 그것을하고있다! {{Salutation}} {{Client Last Name}...
3921,BIL,{{Person Name}}에서 보낸 청구서를 보는 데 도움을 주실 수 있나요?,결제,청구서 확인,물론이죠! {{Salutation}} {{Client Last Name}}의 송장을...
3922,BIL,송장 #37777을 어떻게 찾을 수 있나요?,결제,청구서 확인,번호 #{{Invoice Number}}가 포함된 송장을 찾는 데 혼란을 겪고 계신...
...,...,...,...,...,...
12898,BLMQ,온라인 결제 관련 문제를 알리는 데 도움이 필요합니다.,결제,결제 문제,고객님는 온라인 결제에 문제가 있어 이에 대해 당사에 알리기 위해 도움을 구하고 계...
12899,BLMQWZ,결제 오류를 알리는 데 도움이 필요해요,결제,결제 문제,결제 시 발생한 오류에 대해 저희에게 연락해주셔서 다행입니다. 불편을 끼쳐드린 점 ...
12900,BILMZ,온라인 결제 관련 문제를 어디에 알릴 수 있나요?,결제,결제 문제,온라인 결제 시 발생하는 문제에 대해 당사에 알리려면 전담 고객 지원팀에 문의하세요...
12901,BILMQ,온라인 결제에 문제가 있을 때 어떻게 알릴 수 있나요?,결제,결제 문제,온라인 결제와 관련된 문제를 당사에 알리는 것에 대해 문의해 주셔서 감사합니다. 안...


In [21]:
import pandas as pd
import re

# 1. 템플릿 변수 매핑 테이블 정의
# 자주 등장하는 영어 변수들을 한국어 의미에 맞게 그룹화합니다.
def replace_template_vars(text):
    if pd.isna(text):
        return text
    
    text = str(text)
    
    # 정규표현식 기반 변수 치환 로직
    # 주의: 순서가 중요하며 많이 겹치는 단어는 구체적인 것을 먼저 적어야 합니다.
    
    # 1. 주문/송장/번호 관련
    text = re.sub(r'(?i)\{\{(Order Number|Invoice Number|Tracking Number|Case Number)\}\}', '[주문/송장번호]', text)
    
    # 2. 연락처 및 이메일 관련
    text = re.sub(r'(?i)\{\{(Customer Support Phone Number|Toll-Free Number|Company Phone Number|Customer Support Contact Number)\}\}', '[고객센터전화번호]', text)
    text = re.sub(r'(?i)\{\{(Customer Support Email|Customer Assistance Email|Customer Service Email|Email Address)\}\}', '[이메일주소]', text)
    
    # 3. URL 및 온라인 포털 관련
    text = re.sub(r'(?i)\{\{(Website URL|Online Company Portal Info|Online Order Interaction)\}\}', '[웹사이트/메뉴]', text)
    
    # 4. 시간 및 기간 관련
    text = re.sub(r'(?i)\{\{(Customer Support Hours|Date Range|Business Hours|Customer Service Hours|Working Hours)\}\}', '[운영시간/기간]', text)
    
    # 5. 고객 이름 및 호칭 관련
    text = re.sub(r'(?i)\{\{(Person Name|Client Last Name|Salutation|Client Name|Client First Name|Client Full Name)\}\}', '[고객이름]', text)
    
    # 6. 금액 및 결제 관련
    text = re.sub(r'(?i)\{\{(Refund Amount|Money Amount|Currency Symbol)\}\}', '[금액]', text)
    
    # 7. 주소 및 지역 관련
    text = re.sub(r'(?i)\{\{(Delivery City|Delivery Country|Store Location|Country|Destination)\}\}', '[주소/지역]', text)
    
    # 8. 그 외 모든 중괄호 변수를 [기타정보]로 일괄 치환 (Fallback)
    text = re.sub(r'\{\{.*?\}\}', '[기타정보]', text)
    
    return text

def clean_text_advanced(text):
    if pd.isna(text):
        return text
    text = replace_template_vars(text)
    # 불필요한 줄바꿈 및 공백 제거
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# 2. 데이터 불러오기 및 정제 적용
print("데이터 전처리 시작...")
df = pd.read_csv('cs2_data_ko.csv')

# 중복 제거 (문의 내용 기준)
initial_shape = df.shape[0]
df.drop_duplicates(subset=['문의 내용'], inplace=True)
dedup_shape = df.shape[0]
print(f"중복 제거 완료: {initial_shape}행 -> {dedup_shape}행")

# 결측치 제거
df.dropna(inplace=True)

# 텍스트 정제 적용
df['정제된 문의'] = df['문의 내용'].apply(clean_text_advanced)
df['정제된 응답'] = df['응답'].apply(clean_text_advanced)

# 3. 결과 확인 및 저장
print("\n전처리 샘플 확인:")
print(df[['정제된 문의', '정제된 응답']].head(10))

df.to_csv('1차정제완료_cs데이터.csv', index=False, encoding='utf-8-sig')
print("\n'1차정제완료_cs데이터.csv' 파일로 저장이 완료되었습니다.")

데이터 전처리 시작...
중복 제거 완료: 16891행 -> 10784행

전처리 샘플 확인:
                                       정제된 문의  \
0                       주문 [주문/송장번호] 취소 관련 문의   
1                주문 [주문/송장번호] 취소에 관해 질문이 있습니다   
2                      구매 [주문/송장번호] 취소를 도와주세요   
3                      구매 [주문/송장번호]를 취소해야 합니다   
4   이 주문을 결제할 여유가 없습니다. 구매 [주문/송장번호]를 취소해 주세요   
5                주문 [주문/송장번호] 취소를 도와주실 수 있나요?   
6        더 이상 주문 [주문/송장번호]를 결제할 수 없으니 취소해 주세요   
7                     구매 [주문/송장번호]를 취소하려고 합니다   
8                    구매 [주문/송장번호]를 꼭 취소해야 합니다   
10       주문 [주문/송장번호]를 취소하려면 어떻게 해야 할지 모르겠습니다   

                                               정제된 응답  
0   주문 [주문/송장번호] 취소와 관련해 문의하신 내용을 확인했습니다. 궁금하신 점을 ...  
1   주문 [주문/송장번호] 취소에 관해 궁금한 점이 있으시군요. 구체적인 질문을 말씀해...  
2   구매 번호 [주문/송장번호]의 구매 취소에 도움이 필요하시군요. 불편을 드려 죄송하...  
3   주문 번호 [주문/송장번호]의 구매를 취소하는 데 도움이 필요하신 것으로 확인했습니...  
4   경제적인 사정으로 주문 번호 [주문/송장번호]의 구매를 취소해야 하시는군요. 상황을...  
5   물론입니다. 주문 번호 [주문/송장번호]의 취소를 도와드리겠습니다. 아래 단계를 따...  
6   현재 사정상 주문 [

In [ ]:
import pandas as pd
import re

def process_payment_category(filepath):
    # 1. 데이터 불러오기 및 '결제' 카테고리 필터링
    try:
        df = pd.read_csv(filepath)
    except FileNotFoundError:
        print(f"오류: '{filepath}' 파일을 찾을 수 없습니다. 파일명과 경로를 확인해주세요.")
        return

    df_payment = df[df['카테고리'] == '결제'].copy()
    print(f"초기 '결제' 데이터 수: {df_payment.shape[0]}건")
    
    # 2. 중복 및 결측치 제거
    dp = df_payment.duplicated(subset=['문의 내용']).sum()
    df_payment.drop_duplicates(subset=['문의 내용'], inplace=True)
    dn = df_payment.isnull().any(axis=1).sum()
    df_payment.dropna(inplace=True)
    print(f"중복({dp})/결측치({dn}) 제거 후 데이터 수: {df_payment.shape[0]}건")
    
    # 3. 텍스트 정제 함수 (정규식 에러 방지용 flags=re.I 적용)
    def clean_payment_text(text):
        if pd.isna(text):
            return text
        text = str(text)
        
        # 결제 카테고리 템플릿 변수 치환
        text = re.sub(r'\{\{(Customer Support Phone Number|Payment Issue Phone Number)\}\}', '[고객센터전화번호]', text, flags=re.I)
        text = re.sub(r'\{\{(Customer Support Email|Customer Service Email|Payment Issue Email)\}\}', '[고객센터이메일]', text, flags=re.I)
        text = re.sub(r'\{\{Company Name\}\}', '[회사명]', text, flags=re.I)
        text = re.sub(r'\{\{Website URL\}\}', '[웹사이트]', text, flags=re.I)
        text = re.sub(r'\{\{Customer Support Hours\}\}', '[운영시간]', text, flags=re.I)
        
        # 기타 잔여 변수 일괄 치환
        text = re.sub(r'\{\{.*?\}\}', '[기타정보]', text)
        
        # 공백 및 줄바꿈 압축
        text = re.sub(r'\n+', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        
        return text

    # 4. 정제 함수 적용
    df_payment['정제된 문의'] = df_payment['문의 내용'].apply(clean_payment_text)
    df_payment['정제된 응답'] = df_payment['응답'].apply(clean_payment_text)
    
    # 5. 최종 데이터프레임 구성 및 저장
    df_final = df_payment[['플래그', '정제된 문의', '카테고리', '의도', '정제된 응답']].copy()
    df_final.rename(columns={'정제된 문의': '문의 내용', '정제된 응답': '응답'}, inplace=True)
    
    output_filename = 'cs2_결제카테고리_정제완료.csv'
    df_final.to_csv(output_filename, index=False, encoding='utf-8-sig')
    print(f"전처리 완료: '{output_filename}' 파일이 성공적으로 저장되었습니다.")

# 함수 실행
process_payment_category('cs2_data_ko.csv')

초기 '결제' 데이터 수: 3997건
중복1399/결측치0 제거 후 데이터 수: 2598건
전처리 완료: 'cs2_결제카테고리_정제완료.csv' 파일이 성공적으로 저장되었습니다.


In [1]:
import pandas as pd
import re

def clean_text_general(text):
    if pd.isna(text):
        return text
    text = str(text)
    
    # 1. 주문/송장/번호 관련
    text = re.sub(r'\{\{(Order Number|Invoice Number|Tracking Number|Case Number)\}\}', '[주문/송장번호]', text, flags=re.I)
    # 2. 연락처 및 이메일 관련
    text = re.sub(r'\{\{(Customer Support Phone Number|Toll-Free Number|Company Phone Number|Payment Issue Phone Number)\}\}', '[고객센터전화번호]', text, flags=re.I)
    text = re.sub(r'\{\{(Customer Support Email|Customer Service Email|Payment Issue Email|Email Address)\}\}', '[이메일주소]', text, flags=re.I)
    # 3. URL 및 온라인 포털 관련
    text = re.sub(r'\{\{(Website URL|Online Company Portal Info|Online Order Interaction)\}\}', '[웹사이트/메뉴]', text, flags=re.I)
    # 4. 시간 및 기간 관련
    text = re.sub(r'\{\{(Customer Support Hours|Date Range|Business Hours|Working Hours|Refund Processing Time|Cancellation Refund Time)\}\}', '[운영시간/기간]', text, flags=re.I)
    # 5. 고객 이름 관련
    text = re.sub(r'\{\{(Person Name|Client Last Name|Salutation|Client Name|Client First Name|Client Full Name)\}\}', '[고객이름]', text, flags=re.I)
    # 6. 금액 관련
    text = re.sub(r'\{\{(Refund Amount|Money Amount|Currency Symbol)\}\}', '[금액]', text, flags=re.I)
    # 7. 주소 및 지역 관련
    text = re.sub(r'\{\{(Delivery City|Delivery Country|Store Location|Country|Destination)\}\}', '[주소/지역]', text, flags=re.I)
    # 8. 회사명
    text = re.sub(r'\{\{Company Name\}\}', '[회사명]', text, flags=re.I)
    
    # 그 외 예측하지 못한 모든 잔여 변수 일괄 치환
    text = re.sub(r'\{\{.*?\}\}', '[기타정보]', text)
    
    # 공백 및 줄바꿈 압축
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

def process_and_save_categories(df, target_categories, output_filename):
    # 1. 지정된 카테고리만 필터링 (여러 카테고리를 리스트로 받아 동시 처리 가능)
    df_subset = df[df['카테고리'].isin(target_categories)].copy()
    
    # 2. 중복 및 결측치 제거
    df_subset.drop_duplicates(subset=['문의 내용'], inplace=True)
    df_subset.dropna(inplace=True)
    
    # 3. 정제 함수 적용
    df_subset['정제된 문의'] = df_subset['문의 내용'].apply(clean_text_general)
    df_subset['정제된 응답'] = df_subset['응답'].apply(clean_text_general)
    
    # 4. 컬럼명 원복 및 저장
    df_final = df_subset[['플래그', '정제된 문의', '카테고리', '의도', '정제된 응답']].copy()
    df_final.rename(columns={'정제된 문의': '문의 내용', '정제된 응답': '응답'}, inplace=True)
    
    df_final.to_csv(output_filename, index=False, encoding='utf-8-sig')
    print(f"'{output_filename}' 저장 완료 (데이터 수: {df_final.shape[0]}건)")

# --- 메인 실행부 ---
try:
    print("데이터 전처리 시작...\n")
    df_main = pd.read_csv('cs2_data_ko.csv')
    
    # 조원별 작업 파일 생성
    process_and_save_categories(df_main, ['주문'], 'cs2_주문_정제완료.csv')
    process_and_save_categories(df_main, ['환불'], 'cs2_환불_정제완료.csv')
    process_and_save_categories(df_main, ['배송', '취소'], 'cs2_배송_취소_정제완료.csv')
    process_and_save_categories(df_main, ['배송지', '문의'], 'cs2_배송지_문의_정제완료.csv')
    
    print("\n모든 조원의 데이터 전처리가 성공적으로 끝났습니다.")

except FileNotFoundError:
    print("오류: 'cs2_data_ko.csv' 파일을 찾을 수 없습니다.")

데이터 전처리 시작...

'cs2_주문_정제완료.csv' 저장 완료 (데이터 수: 2539건)
'cs2_환불_정제완료.csv' 저장 완료 (데이터 수: 1774건)
'cs2_배송_취소_정제완료.csv' 저장 완료 (데이터 수: 1974건)
'cs2_배송지_문의_정제완료.csv' 저장 완료 (데이터 수: 1899건)

모든 조원의 데이터 전처리가 성공적으로 끝났습니다.
